# 4. Feature engineering: select, calculate and inspect

Run the three representation notebooks first. This notebook builds ranking inputs using static content and historical state available **before each request’s feedback**.

**Reading path:** choose features → review calculations → follow one worked example and its checks → build/reuse the dataset → inspect results. The optional appendix contains additional demonstrations.

The core calculations stay visible here. Shared helpers handle input/cache validation, storage and reports. Teaching inputs stay in memory; the saved feature dataset uses the real prepared MIND inputs.

## 0. Setup and run controls

Core calculations remain visible here. Imported helpers handle file/cache validation, keyed input checks, storage and reports. You do not run those helper files separately.

`REBUILD=False` reuses saved compatible outputs. After editing a calculation, use `REBUILD=True` and rebuild dependent datasets. The examples always execute, even when the dataset itself is loaded from cache.

Use `FULL_RUN=False` in the first code cell for a 1,000-timestamp smoke run, or `True` for full interactive execution. Terminal flags take priority. Notebook outputs refresh on rerun; smoke/full datasets use separate folders and `REBUILD=False` preserves a compatible saved dataset.

In [1]:
# Direct notebook: False = first 1,000 timestamps; True = full dataset.
# run_all.py flags take priority over this notebook preference.
FULL_RUN = False

from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Separate code location from the optional data/output root.
CODE_ROOT = (
    Path(os.environ["NEWSREC_CODE"]).expanduser().resolve()
    if os.environ.get("NEWSREC_CODE")
    else next(
        (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/newsrec").is_dir()),
        None,
    )
)
if CODE_ROOT is None or not (CODE_ROOT / "src/newsrec").is_dir():
    raise FileNotFoundError(
        "Open the project checkout, or set NEWSREC_CODE to the folder containing src/newsrec/."
    )
sys.path.insert(0, str(CODE_ROOT / "src"))
from newsrec.common import Settings, saved_stage_ready

DATA_ROOT = (
    None  # Optional: point to another folder holding the project's saved datasets.
)
ROOT = (
    Path(DATA_ROOT or os.environ.get("NEWSREC_ROOT") or CODE_ROOT)
    .expanduser()
    .resolve()
)
ARTIFACTS = ROOT / "3_Representations/artifacts"
print("Data/output root:", ROOT)

from newsrec.io import feature_run, feature_sample
import hashlib
from collections import deque
from newsrec.common import DEFAULT_SETTINGS
from newsrec.temporal import TemporalState as _StateStorage, User, HOUR
from newsrec.common import prepare_feature_pairs, validate_feature_schema
from newsrec.representations import Bundle
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.common import normalise
from dataclasses import replace
from types import SimpleNamespace

Data/output root: /Users/bryan03/Desktop/BT4222-Group-6-


In [2]:
# Defaults live in src/newsrec/common.py; override with Settings(...) if needed.
SETTINGS = Settings()
REBUILD = False  # Reuse saved outputs; True regenerates this notebook's results.
# run_all.py --force overrides the preference for this execution only.
REBUILD = REBUILD or os.environ.get("NEWSREC_FORCE_REBUILD") == "1"

display(
    pd.DataFrame(
        {
            "setting": [
                "history_cap",  # Retained sequence references; vectors pool the full causal history
                "recent_count",  # Number of recent items to consider
                "recent_hours",  # Time window in hours for recent items
                "interest_count",  # Number of interest items to consider
                "negative_count",  # Number of negative items to consider
                "context_features",  # Boolean: enable optional slate-relative features
                "update_mode",
            ],
            "value": [
                SETTINGS.history_cap,
                SETTINGS.recent_count,
                SETTINGS.recent_hours,
                SETTINGS.interest_count,
                SETTINGS.negative_count,
                SETTINGS.context_features,
                SETTINGS.update_mode,
            ],
        }
    )
)

,setting,value
0,history_cap,50
1,recent_count,10
2,recent_hours,24
3,interest_count,3
4,negative_count,4
5,context_features,False
6,update_mode,prequential


## 1. Choose model inputs

Only `schema["model_columns"]` enters a ranker. The catalogue below is the complete editable list; the overview in 1.1 explains the groups without printing those lists again. Selected model values are stored as `float32`.

Missing measurements use zero plus validity/missingness flags. Features named `log_...` already use `log1p`. Fit any later scaler or ranker on training rows only.

### 1.0 Edit features here

1. **Choose groups:** edit `ENABLED_GROUPS` in the controls cell. Keep only the groups you want a ranker to use.
2. **Disable individual features:** add their exact names to `DISABLED_FEATURES`.
3. **Add a feature:** add an entry to `EXTRA_FEATURES` with a group, definition and `compute(frame, cfg)` calculation. The commented example is ready to uncomment. For a new group name, also add it to `ENABLED_GROUPS`.
4. Rerun from here, inspect the examples, then set `REBUILD=True` to regenerate the real feature dataset. Restore `REBUILD=False` afterward to reuse it.

The saved schema contains your selected model columns. Unselected built-in values can remain in the wide output for coverage, diagnostics or later ablations; they must not enter the model. Slate context additionally requires `SETTINGS.context_features=True`.

Changing selection/schema is detected when opening a cache. After editing a formula without changing its name/definition, explicitly set `REBUILD=True`. Use only static content or strictly earlier information when writing new calculations; missing values need a suitable flag.

In [ ]:
# Prior logged statistics use allowed feedback strictly before the request timestamp.
# Supplied histories have no click timestamps; they seed profiles but not logged exposure counters.
GROUPS = {
    "content": [  # What the article is: category, text length, entity coverage, embedding availability
        "category_code",  # Training-fitted category ID; 0 means unknown to the training mapping (categorical, not ordinal)
        "subcategory_code",  # Training-fitted subcategory ID; 0 means unknown to the training mapping (categorical, not ordinal)
        "title_word_count",  # Whitespace-separated word count after title text cleaning
        "abstract_word_count",  # Whitespace-separated word count after abstract text cleaning; 0 if empty
        "has_abstract",  # 1 if the cleaned abstract is non-empty, even if its embedding is unusable
        "n_annotated_entities",  # Number of distinct parsed entity IDs across title and abstract; not the embedding dimension
        "entity_coverage",  # Distinct annotated IDs with nonzero lookup vectors / distinct annotated IDs; 0 if none
        "article_has_text",  # 1 if the candidate has a nonzero fused text vector
        "article_has_entity",  # 1 if the candidate has a nonzero pooled entity vector
    ],
    "history": [  # Full causal history = supplied snapshot + allowed earlier clicks; repeats retained, sequence cap only
        "log_history_length",  # log1p(supplied-history items + allowed earlier clicked events); repeated clicks count
        "retained_history_length",  # Number of nonpadding sequence items retained, at most cfg.history_cap; pooling is uncapped
        "valid_text_history_fraction",  # Fraction of full causal history entries with nonzero text vectors; repeats count, 0 if empty
        "valid_entity_history_fraction",  # Fraction of full causal history entries with nonzero entity vectors; repeats count, 0 if empty
        "no_supplied_history",  # 1 if the original supplied-history snapshot is empty, even if later observed clicks exist
        "no_causal_history",  # 1 if supplied history plus allowed earlier observed clicks contains no items
        "no_text_profile",  # 1 if the full causal pooled text vector is zero/unusable
        "no_entity_profile",  # 1 if the full causal pooled entity vector is zero/unusable
        "no_recent_profile",  # 1 if no usable pooled recent text vector exists
    ],
    "match": [  # Candidate matches to full causal, supplied-only and timestamped recent user views
        "category_affinity",  # Candidate-category count in full causal click history / full history length; 0 if empty
        "subcategory_affinity",  # Candidate-subcategory count in full causal click history / full history length; 0 if empty
        "seen_category",  # 1 if the candidate category occurs in supplied history or allowed earlier clicks
        "text_cosine",  # Candidate fused text vs full causal pooled text (supplied history + allowed earlier clicks)
        "entity_cosine",  # Candidate entity vector vs full causal pooled entity vector
        "long_term_cosine",  # Candidate fused text vs supplied-history-only pooled text; always text, not entity
        "recent_cosine",  # Candidate fused text vs earlier timestamped-click text; cfg.recent_hours / cfg.recent_count (24h / 10 by default)
        "text_match_valid",  # 1 if both the candidate text vector and full causal text profile are nonzero
        "entity_match_valid",  # 1 if both the candidate entity vector and full causal entity profile are nonzero
        "long_term_match_valid",  # 1 if both the candidate text vector and supplied-only text profile are nonzero
        "recent_match_valid",  # 1 if both the candidate text vector and recent text profile are usable
    ],
    "repeat": [  # Allowed earlier logged user/article interactions; previously_clicked also checks the supplied snapshot
        "log_prior_exposure_count",  # log1p of this user/article pair's allowed earlier logged exposures; excludes supplied-history-only interactions
        "log_prior_nonclick_count",  # log1p(prior logged pair exposures - prior logged pair clicks)
        "log_prior_pair_click_count",  # log1p of allowed earlier logged clicks on this user/article pair; supplied-only clicks are excluded
        "hours_since_last_exposure",  # Hours since the last allowed earlier logged pair exposure; 0 if no_prior_pair_exposure = 1
        "no_prior_pair_exposure",  # 1 if this user/article pair has no allowed earlier logged exposure; supplied history does not record exposures
        "previously_clicked",  # 1 if the article is in the supplied snapshot OR has an allowed earlier logged click by this user
    ],
    "popularity": [  # Allowed earlier article statistics across users and support flags
        "log_prior_article_exposures",  # log1p of allowed earlier article exposures across all users
        "log_prior_article_clicks",  # log1p of allowed earlier article clicks across all users
        "smoothed_prior_article_ctr",  # (article clicks + cfg.ctr_strength * global_prior) / (article exposures + cfg.ctr_strength); Beta(1,1) smooths global_prior
        "article_low_support",  # 1 if prior article exposures < cfg.article_ctr_min_exposures (100 by default)
        "population_startup",  # 1 if no allowed population exposure has been recorded; excluded by default
    ],
    "freshness": [  # How long the article has been observed (age since first exposure, not publication)
        "article_age_hours",  # Hours since first allowed earlier logged exposure; 0 if unseen; not publication age
        "no_prior_article_exposure",  # 1 if the article has no allowed earlier logged exposure across users
        "age_left_censored",  # 1 only if first prior exposure equals replay start; narrow boundary diagnostic, excluded by default
    ],
    "multi_interest": [  # Best title-vector cosine over selected, usable full-history category interests
        "multi_interest_match_score",  # Maximum candidate-title cosine over usable full-history category interests; up to cfg.interest_count (default 3), 0 if none
        "interest_group_count",  # Number of selected category interests with nonzero pooled title vectors; these enter the maximum
        "selected_interest_group_count",  # Number of top-count categories considered BEFORE filtering unusable vectors; may exceed interest_group_count
        "multi_interest_valid",  # 1 if at least one usable category-interest vector AND a nonzero candidate title vector exist
    ],
    "half_life": [  # Observed smoothed-CTR decay between completed windows; unsupported estimates are masked
        "estimated_article_half_life_hours",  # ln(2) * gap_hours / ln(early_CTR / recent_CTR); equal completed 6h windows, 0 if invalid/unsupported
        "has_half_life",  # 1 for a positive finite decay estimate from supported, nonoverlapping completed windows with declining smoothed CTR
        "stale_after_grace",  # 1 if has_half_life AND article_age_hours > estimated_article_half_life_hours + 6
    ],
    "request_context": [  # User request gaps plus optional log-clock calendar buckets (calendar fields excluded by default)
        "day_part",  # Request log-clock hour // 6: 0=00-05, 1=06-11, 2=12-17, 3=18-23; excluded by default
        "weekday",  # Request log-clock weekday: Monday=0 through Sunday=6; excluded by default
        "hours_since_previous_impression",  # Hours since this user's previous replayed impression timestamp; 0 if first_visit = 1
        "first_visit",  # 1 if no earlier impression for this user has been replayed; supplied history may still exist
    ],
    "slate_context": [  # Optional comparisons within the actual candidate set for each impression; disabled by default
        "personalized_relative_advantage",  # Multi-interest score minus the actual candidate-set median for this impression; 0 if slate_context_valid = 0
        "slate_context_valid",  # 1 if EVERY actual candidate in the impression has multi_interest_valid = 1
    ],
}

# Schema description strings below are retained so existing saved outputs remain compatible.
# "Up to three" describes the default; calculations use cfg.interest_count.
# For half-life, 6h is each window's width; the centre-to-centre gap is measured in hours.
# global_prior = (prior total clicks + 1) / (prior total exposures + 2); the article uses cfg.ctr_strength.
FORMULAS = {
    "category_affinity": "full prior clicked category count / full history count; zero if empty",
    "subcategory_affinity": "full prior clicked subcategory count / full history count; zero if empty",
    "seen_category": "prior candidate-category count > 0",
    "smoothed_prior_article_ctr": "(prior article clicks + strength * expanding Beta(1,1) global CTR) / (prior article exposures + strength)",
    "article_age_hours": "hours since first prior observed exposure; zero with flag if none",
    "estimated_article_half_life_hours": "ln(2) * completed 6h window-centre gap / ln(early CTR / recent CTR); zero plus mask when unsupported",
    "multi_interest_match_score": "max title64 cosine over up to three valid category interest vectors",
    "personalized_relative_advantage": "candidate multi-interest match minus actual candidate-set median; zero if context invalid",
    "article_low_support": "prior article exposures < cfg.article_ctr_min_exposures",
    "age_left_censored": "boundary-only diagnostic: first prior exposure equals log start; does not identify all pre-existing articles",
    "population_startup": "no population exposures observed before request; diagnostic by default",
}

In [4]:
# Choose model inputs here. Removing a group excludes it from model_columns.
ENABLED_GROUPS = [
    "content",
    "history",
    "match",
    "repeat",
    "popularity",
    "freshness",
    "multi_interest",
    "half_life",
    "request_context",
    "slate_context",
    "custom",
]
# Optional ablations: remove a name here to re-enable it.
# Friday is absent from training; calendar buckets need separate validation.
# Boundary/startup flags are retained as diagnostics, not default model inputs.
DISABLED_FEATURES = ["weekday", "day_part", "population_startup", "age_left_censored"]

# Add a feature by registering its name, explanation and calculation in this cell.
# The frame contains already-computed causal features/diagnostics, never current labels.
EXTRA_FEATURES = {
    # "prior_click_share": {
    #     "group": "custom",
    #     "definition": "past article clicks / (past article exposures + 1)",
    #     "compute": lambda frame, cfg: frame["prior_article_clicks"]
    #                                    / (frame["prior_article_exposures"] + 1),
    # },
}

### Resolve the selected contract

This function supplies the actual saved schema and model allowlist to the builder. It rejects unknown groups, duplicate names, empty selections and protected key/target/diagnostic fields. You normally edit the controls above rather than this function.

In [5]:
def feature_schema(cfg=DEFAULT_SETTINGS):
    known_groups = (
        set(GROUPS)
        | {"custom"}
        | {s.get("group", "custom") for s in EXTRA_FEATURES.values()}
    )
    if set(ENABLED_GROUPS) - known_groups:
        raise ValueError(
            f"Unknown feature groups: {sorted(set(ENABLED_GROUPS) - known_groups)}"
        )
    builtin_names = {name for names in GROUPS.values() for name in names}
    if set(EXTRA_FEATURES) & builtin_names:
        raise ValueError("Custom feature names must not overwrite built-in features")
    if set(DISABLED_FEATURES) - (builtin_names | set(EXTRA_FEATURES)):
        raise ValueError("DISABLED_FEATURES contains an unknown feature name")
    features = []
    for group, names in GROUPS.items():
        if group not in ENABLED_GROUPS:
            continue
        if group == "slate_context" and (not cfg.context_features):
            continue
        for name in names:
            if name in DISABLED_FEATURES:
                continue
            features.append(
                {
                    "name": name,
                    "dtype": "float32",
                    "kind": "categorical" if name in {"category_code", "subcategory_code", "day_part", "weekday"} else "numeric",
                    "group": group,
                    "missing_value": 0,
                    "definition": FORMULAS.get(name, name.replace("_", " ")),
                    "cutoff": "static train-fitted content or events strictly before request timestamp",
                    "preprocessing": "integer category ID; use embedding/one-hot, never scale as an ordinal number; 0 is unknown for content codes"
                    if name in {"category_code", "subcategory_code", "day_part", "weekday"}
                    else "log1p already applied"
                    if name.startswith("log_")
                    else "none; fit any scaler on training rows only",
                    "consumer": "ranker; slate context requires complete actual candidate set"
                    if group == "slate_context"
                    else "ranker",
                }
            )
    for name, specification in EXTRA_FEATURES.items():
        group = specification.get("group", "custom")
        if group not in ENABLED_GROUPS or name in DISABLED_FEATURES:
            continue
        if not callable(specification.get("compute")):
            raise ValueError(
                f"Custom feature {name} needs a compute(frame, cfg) function"
            )
        features.append(
            {
                "name": name,
                "dtype": "float32",
                "group": group,
                "missing_value": 0,
                "definition": specification["definition"],
                "cutoff": "static train-fitted content or events strictly before request timestamp",
                "preprocessing": "none; fit any scaler on training rows only",
                "consumer": "ranker",
            }
        )
    return validate_feature_schema(
        {
            "version": "ranking-features-v2-full-causal-pooling",
            "features": features,
            "model_columns": [f["name"] for f in features],
            "categorical_columns": [f["name"] for f in features if f.get("kind") == "categorical"],
            "profile_pooling": "full supplied snapshot plus every earlier observed click; cap sequence references only",
            "keys": ["impression_uid", "news_code"],
            "diagnostics": [
                "user_code",
                "article_row",
                "timestamp",
                "state_version",
                "candidate_source",
                "eligible",
                "history_length",
                "prior_article_exposures",
                "prior_article_clicks",
                "prior_exposure_count",
                "half_life_reason",
                "half_life_early_support",
                "half_life_recent_support",
            ],
            "excluded": [
                "clicked",
                "position",
                "slate_len",
                "n_clicks",
                "n_negatives",
                "split",
                "user_code",
                "news_code",
                "impression_uid",
            ],
            "update_mode": cfg.update_mode,
            "config_hash": cfg.fingerprint(),
        }
    )

In [6]:
schema = feature_schema(SETTINGS)
contract = pd.DataFrame(schema["features"])
display(
    contract.groupby("group", sort=False).size().rename("selected_features").to_frame()
)
print("Total selected model features:", len(schema["model_columns"]))
custom_view = contract.loc[
    contract["name"].isin(EXTRA_FEATURES), ["name", "group", "definition"]
]
if not custom_view.empty:
    display(custom_view.reset_index(drop=True))

,selected_features
group,
content,9
history,9
match,11
repeat,6
popularity,4
freshness,2
multi_interest,4
half_life,3
request_context,2


Total selected model features: 50


### 1.1 Feature-group overview

| Group | What it contributes | Example |
|---|---|---|
| Content | Article metadata and representation availability | `title_word_count` |
| History | Amount and usability of causal user history | `retained_history_length` |
| Match | User–article affinity and cosine similarities | `text_cosine` |
| Repeat exposure | Earlier views/non-clicks/clicks for this user–article pair | `log_prior_exposure_count` |
| Popularity | Earlier article exposure/click counts and smoothed CTR | `smoothed_prior_article_ctr` |
| Freshness | Time since first prior observed exposure; not publication age | `article_age_hours` |
| Multiple interests | Best match across category-specific interest vectors | `multi_interest_match_score` |
| Half-life | Supported decay estimated from completed past windows | `has_half_life` |
| Request context | Clock and previous-visit information | `hours_since_previous_impression` |
| Slate context | Interest score relative to the actual candidate set | `personalized_relative_advantage` |
| Custom | Your additional registered calculations | See `EXTRA_FEATURES` |

The counts above reflect your current selection. Slate context additionally requires `SETTINGS.context_features=True` and must use the full actual candidate set before training sampling. Its value can change if retrieval changes that set.

Defaults exclude `weekday`, `day_part`, `population_startup`, and `age_left_censored`. Calendar buckets need an ablation; Friday is absent from training. Category/subcategory IDs are categorical: use train-fitted one-hot encoding or embeddings, not numeric scaling. The schema records `categorical_columns`.

`age_left_censored` is retained only as a boundary diagnostic (first prior exposure at exact log start). It cannot identify all articles that existed earlier. No publication timestamps are available; a 24–48-hour window would be a heuristic, not a verified censoring label.

### 1.2 Model-input safeguards

- **Keys and diagnostics stay separate:** use IDs to join predictions; use raw counts/state versions/reason strings for audits, not as implicit model inputs.
- **Targets stay separate:** `clicked`, shuffled position and current slate/click summaries are outside the model allowlist. Unshown retrieved items do not receive invented labels.
- **Score before feedback:** every request at timestamp `t`, including a train/validation boundary tie, sees the same pre-feedback state. Apply allowed feedback only after scoring the complete batch.
- **Specify evaluation mode:** prequential mode can consume earlier validation/test feedback; frozen mode withholds its target split’s feedback. Neither uses current or future feedback for the current request.

## 2. Define the calculations

These definitions are required by the real builder, the functions remain in the notebook so their formulas can be reviewed and edited directly. One shared example follows all definitions in 2.7, with consolidated checks in 2.8.

### 2.1 Pool valid vectors

`normalised_mean` ignores zero/missing vectors, averages the valid entries, normalises the result, and returns the valid-entry count. A zero output must be interpreted with its validity flag.

In [7]:
def normalised_mean(matrix, rows):
    if not len(rows):
        return (np.zeros(matrix.shape[1], dtype=np.float32), 0)
    selected = matrix[np.asarray(rows, dtype=np.int32)]
    valid = np.linalg.norm(selected, axis=1) > 1e-12
    if valid.any():
        pooled = normalise(selected[valid].mean(axis=0))
    else:
        pooled = np.zeros(matrix.shape[1], np.float32)
    return (pooled, int(valid.sum()))

### 2.2 Maintain causal user profiles

`begin` seeds supplied history once. `profile` pools text/entity and category-interest vectors from the **full causal history**, including repeated earlier clicks. Only the padded sequence references are capped; the supplied-only long-term and timestamped recent views remain separate. `finish` consumes allowed logged feedback after the timestamp has been scored.

Supplied history has no click timestamps, so only timestamped logged clicks enter the recent view. When supplied-history order is unverified, its capped seed uses deterministic ordering rather than claiming recency. Genuine repeat clicks remain events. Shared inheritance provides state allocation, SQLite pair storage and cleanup.

In [8]:
class CausalProfiles(_StateStorage):
    """Profile calculations with shared array/SQLite allocation and cleanup."""

    def begin(self, requests, t):
        t = pd.Timestamp(t)
        if self.active_time is not None:
            raise ValueError("Previous timestamp has not been finished")
        if self.last_update is not None and t <= self.last_update:
            raise ValueError(
                "Replay timestamps must increase strictly; merge same-time requests"
            )
        if t < self.start or t > self.end:
            raise ValueError("Request outside state horizon")
        if (
            requests.impression_uid.duplicated().any()
            or not requests.timestamp.eq(t).all()
        ):
            raise ValueError("begin requires unique requests from this timestamp only")
        self.active_time = t
        for row in requests.itertuples(index=False):
            text = "" if pd.isna(row.history) else str(row.history)
            codes = tuple(int(token.removeprefix("N")) for token in text.split())
            key = int(row.user_code)
            if key in self.users:
                if self.users[key].snapshot_key != codes:
                    raise ValueError(f"Supplied history changed for user {key}")
                continue
            snapshot = self.bundle.rows(codes)
            user = User(snapshot, codes, deque(maxlen=max(self.cfg.history_cap, self.cfg.recent_count)))
            user.categories.update(self.cat[snapshot])
            user.subcategories.update(self.subcat[snapshot])
            self.users[key] = user

    def profile(self, user_code, t):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError(
                "Call begin() for this timestamp before requesting profiles"
            )
        key = int(user_code)
        user = self.users[key]
        cache_key = (key, user.version, t.value)
        if cache_key in self.profiles:
            return self.profiles[cache_key]
        observed = list(user.observed)
        seed = (
            user.snapshot
            if self.cfg.snapshot_order_verified
            else np.random.default_rng(self.cfg.seed + key).permutation(np.sort(user.snapshot))
        )
        history = np.r_[seed, [a for _, a in observed]].astype(np.int32)[
            -self.cfg.history_cap :
        ]
        recent = [
            a for time, a in observed if t.value - time <= self.cfg.recent_hours * HOUR
        ][-self.cfg.recent_count :]
        # Pool the full causal multiset, preserving repeated clicks. Sequence
        # truncation below must not choose which articles define user interests.
        full_history = np.r_[user.snapshot, user.clicked_rows].astype(np.int32)
        text, valid_text = normalised_mean(self.bundle.text, full_history)
        entity, valid_entity = normalised_mean(self.bundle.entity, full_history)
        long, _ = normalised_mean(self.bundle.text, user.snapshot)
        recent_vector, valid_recent = normalised_mean(self.bundle.text, recent)
        interests = []
        interest_categories = []
        for cat, _ in sorted(user.categories.items(), key=lambda x: (-x[1], x[0]))[
            : self.cfg.interest_count
        ]:
            rows = full_history[self.cat[full_history] == cat]
            vector, valid = normalised_mean(self.bundle.title, rows)
            if valid and np.linalg.norm(vector) > 0:
                interests.append(vector)
                interest_categories.append(cat)
        padded = np.zeros(self.cfg.history_cap, np.int32)
        padded[: len(history)] = history
        result = {
            "text": text,
            "entity": entity,
            "long_text": long,
            "recent_text": recent_vector,
            "interests": np.asarray(interests, np.float32).reshape(
                -1, self.cfg.text_dim
            ),
            "history_rows": padded,
            "full_history_rows": full_history,
            "history_mask": padded != 0,
            "selected_interest_categories": interest_categories,
            "recent_rows": list(map(int, recent)),
            "history_length": len(user.snapshot) + user.n_observed,
            "retained_length": len(history),
            "valid_text_fraction": valid_text / max(1, len(full_history)),
            "valid_entity_fraction": valid_entity / max(1, len(full_history)),
            "has_text": bool(np.linalg.norm(text) > 0),
            "has_entity": bool(np.linalg.norm(entity) > 0),
            "has_recent": bool(valid_recent and np.linalg.norm(recent_vector) > 0),
            "no_supplied_history": len(user.snapshot) == 0,
            "group_count": len(interests),
            "selected_group_count": min(self.cfg.interest_count, len(user.categories)),
            "state_version": user.version,
        }
        self.profiles[cache_key] = result
        return result

    def finish(self, requests, events, t, feedback_splits):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError("Timestamp is not active")
        if not events.timestamp.eq(t).all() or not set(events.impression_uid) <= set(
            requests.impression_uid
        ):
            raise ValueError("Feedback must belong to this timestamp's requests")
        owners = requests.set_index("impression_uid").user_code
        if not np.array_equal(
            events.user_code.to_numpy(), events.impression_uid.map(owners).to_numpy()
        ):
            raise ValueError("Feedback/request user mismatch")
        if events.duplicated(["impression_uid", "news_code"]).any():
            raise ValueError("Duplicate feedback event")
        for u in requests.user_code.unique():
            self.users[int(u)].last_impression = t
        selected = events[events.split.isin(feedback_splits)]
        if len(selected):
            if not selected.clicked.isin([0, 1]).all():
                raise ValueError("Feedback labels must be binary")
            rows = self.bundle.rows(selected.news_code)
            labels = selected.clicked.to_numpy(np.int64)
            self.pairs.update(selected.user_code, rows, labels, t)
            np.add.at(self.exposures, rows, 1)
            np.add.at(self.clicks, rows, labels)
            unseen = self.first[rows] < 0
            self.first[rows[unseen]] = t.value
            hour = int(t.value // HOUR - self.start_hour)
            np.add.at(self.hour_exposures[:, hour], rows, 1)
            np.add.at(self.hour_clicks[:, hour], rows, labels)
            self.total_exposures += len(rows)
            self.total_clicks += int(labels.sum())
            clicked = selected.loc[selected.clicked.eq(1)].sort_values(
                ["user_code", "news_code", "impression_uid"]
            )
            for event in clicked.itertuples(index=False):
                user = self.users[int(event.user_code)]
                a = self.bundle.lookup[int(event.news_code)]
                user.observed.append((t.value, a))
                user.clicked_rows.append(a)
                user.n_observed += 1
                user.categories[self.cat[a]] += 1
                user.subcategories[self.subcat[a]] += 1
                user.version += 1
        self.profiles.clear()
        self.last_update = t
        self.active_time = None

### 2.3 Add a supported, past-only decay estimate

`TemporalState` adds half-life to the profile state. It compares two disjoint completed six-hour windows using Beta(1,1) smoothed CTR and configurable minimum exposure support in both windows. The estimate is `ln(2) × window gap / ln(early CTR / recent CTR)`.

No prior exposure, insufficient support or no decline produces zero plus a validity/reason flag. This is observed CTR decay, not publication lifetime or a hard expiry rule. The appendix demonstrates the window boundaries.

In [9]:
class TemporalState(CausalProfiles):
    """Add a past-only freshness estimator to the causal profile state."""

    def half_life(self, rows, t):
        t = pd.Timestamp(t)
        recent_end = int(t.value // HOUR - self.start_hour)
        recent_start = recent_end - 6
        result = []
        for row in rows:
            first = self.first[row]
            if first < 0:
                result.append((0.0, False, "no_prior_exposure", 0, 0))
                continue
            early_start = int((first + HOUR - 1) // HOUR - self.start_hour)
            early_end = early_start + 6
            if recent_start < early_end:
                result.append((0.0, False, "young_or_overlapping", 0, 0))
                continue
            early_e = int(self.hour_exposures[row, early_start:early_end].sum())
            recent_e = int(self.hour_exposures[row, recent_start:recent_end].sum())
            if min(early_e, recent_e) < self.cfg.half_life_min_exposures:
                result.append((0.0, False, "low_support", early_e, recent_e))
                continue
            early_c = int(self.hour_clicks[row, early_start:early_end].sum())
            recent_c = int(self.hour_clicks[row, recent_start:recent_end].sum())
            early = (early_c + 1) / (early_e + 2)
            recent = (recent_c + 1) / (recent_e + 2)
            if recent >= early:
                result.append((0.0, False, "no_decline", early_e, recent_e))
                continue
            estimate = np.log(2) * (recent_start - early_start) / np.log(early / recent)
            ok = bool(np.isfinite(estimate) and estimate > 0)
            result.append(
                (
                    float(estimate) if ok else 0.0,
                    ok,
                    "available" if ok else "invalid",
                    early_e,
                    recent_e,
                )
            )
        return result

### 2.4 Calculate article and pair statistics

`article_features` combines static content with strictly earlier pair exposures, article popularity, observed age and half-life. Smoothed article CTR uses the expanding past-only global CTR. At population startup, the Beta(1,1) global prior is 0.5; explicit flags distinguish that fallback from supported evidence.

In [10]:
def article_features(pairs, rows, state, bundle, cfg, t, candidate_source="retrieved"):
    """Static content and strictly prior pair/article statistics."""
    prior_pairs = state.pairs.lookup(pairs.user_code, rows)
    unique_rows, inverse = np.unique(rows, return_inverse=True)
    unique_half = state.half_life(unique_rows, t)
    half = [unique_half[i] for i in inverse]
    global_prior = (state.total_clicks + 1) / (state.total_exposures + 2)
    metadata = bundle.meta.loc[rows].reset_index(drop=True)
    quality = bundle.quality.loc[pairs.news_code.to_numpy()].reset_index(drop=True)
    article_e = state.exposures[rows]
    article_c = state.clicks[rows]
    first = state.first[rows]
    seen = first >= 0
    pair_e = np.asarray([x[0] for x in prior_pairs], np.int64)
    pair_c = np.asarray([x[1] for x in prior_pairs], np.int64)
    pair_last = np.asarray(
        [-1 if x[2] is None else x[2] for x in prior_pairs], np.int64
    )
    h = np.asarray([x[0] for x in half], np.float64)
    has_h = np.asarray([x[1] for x in half], bool)
    text_valid = np.linalg.norm(bundle.text[rows], axis=1) > 0
    entity_valid = np.linalg.norm(bundle.entity[rows], axis=1) > 0
    title_valid = np.linalg.norm(bundle.title[rows], axis=1) > 0
    ages = np.where(seen, (t.value - first) / HOUR, 0)
    # Raw counts stay diagnostic; log counts, CTR, age and masks feed the ranker.
    values = {
        "impression_uid": pairs.impression_uid.to_numpy(np.int64),
        "news_code": pairs.news_code.to_numpy(np.int64),
        "user_code": pairs.user_code.to_numpy(np.int64),
        "article_row": rows,
        "timestamp": t,
        "candidate_source": candidate_source,
        "eligible": True,
        "category_code": metadata.category_code.to_numpy(),
        "subcategory_code": metadata.subcategory_code.to_numpy(),
        "title_word_count": metadata.title_word_count.to_numpy(),
        "abstract_word_count": metadata.abstract_word_count.to_numpy(),
        "has_abstract": metadata.has_abstract.to_numpy(),
        "n_annotated_entities": quality.n_annotated_entities.to_numpy(),
        "entity_coverage": quality.entity_coverage.to_numpy(),
        "article_has_text": text_valid,
        "article_has_entity": entity_valid,
        "prior_exposure_count": pair_e,
        "log_prior_exposure_count": np.log1p(pair_e),
        "log_prior_nonclick_count": np.log1p(pair_e - pair_c),
        "log_prior_pair_click_count": np.log1p(pair_c),
        "hours_since_last_exposure": np.where(
            pair_last >= 0, (t.value - pair_last) / HOUR, 0
        ),
        "no_prior_pair_exposure": pair_last < 0,
        "prior_article_exposures": article_e,
        "prior_article_clicks": article_c,
        "log_prior_article_exposures": np.log1p(article_e),
        "log_prior_article_clicks": np.log1p(article_c),
        "smoothed_prior_article_ctr": (article_c + cfg.ctr_strength * global_prior)
        / (article_e + cfg.ctr_strength),
        "article_low_support": article_e < cfg.article_ctr_min_exposures,
        "population_startup": state.total_exposures == 0,
        "article_age_hours": ages,
        "no_prior_article_exposure": ~seen,
        "age_left_censored": first == state.start.value,
        "estimated_article_half_life_hours": h,
        "has_half_life": has_h,
        "stale_after_grace": has_h & (ages > h + 6),
        "half_life_reason": [x[2] for x in half],
        "half_life_early_support": [x[3] for x in half],
        "half_life_recent_support": [x[4] for x in half],
        "day_part": t.hour // 6,
        "weekday": t.weekday(),
    }
    return values, {
        "text": text_valid,
        "entity": entity_valid,
        "title": title_valid,
        "prior_pair_clicks": pair_c,
    }

### 2.5 Add user–article matching

`add_user_features` reads the causal profiles to calculate history coverage, category affinity, long-term/recent cosine similarity and repeated-click indicators. Multi-interest matching takes the best title-space cosine across valid category interests; it can disagree with a single averaged profile. This function does not consume labels or mutate history.

In [11]:
def add_user_features(values, pairs, rows, state, bundle, cfg, t, validity):
    """Add profile matches without consuming any current labels or changing state."""
    n = len(pairs)
    pair_c = validity["prior_pair_clicks"]
    text_valid, entity_valid, title_valid = (
        validity[k] for k in ("text", "entity", "title")
    )
    user_columns = [
        "state_version",
        "history_length",
        "log_history_length",
        "retained_history_length",
        "valid_text_history_fraction",
        "valid_entity_history_fraction",
        "no_supplied_history",
        "no_causal_history",
        "no_text_profile",
        "no_entity_profile",
        "no_recent_profile",
        "category_affinity",
        "subcategory_affinity",
        "seen_category",
        "text_cosine",
        "entity_cosine",
        "long_term_cosine",
        "recent_cosine",
        "previously_clicked",
        "multi_interest_match_score",
        "interest_group_count",
        "selected_interest_group_count",
        "multi_interest_valid",
        "hours_since_previous_impression",
        "first_visit",
        "text_match_valid",
        "entity_match_valid",
        "long_term_match_valid",
        "recent_match_valid",
    ]
    values.update({name: np.zeros(n, np.float64) for name in user_columns})
    for user_code, positions in pairs.groupby("user_code", sort=False).indices.items():
        idx = np.asarray(positions)
        article_rows = rows[idx]
        user = state.users[int(user_code)]
        p = state.profile(user_code, t)
        total = p["history_length"]
        category_counts = np.asarray(
            [user.categories[c] for c in state.cat[article_rows]]
        )
        subcategory_counts = np.asarray(
            [user.subcategories[c] for c in state.subcat[article_rows]]
        )
        dynamic = {
            "state_version": p["state_version"],
            "history_length": total,
            "log_history_length": np.log1p(total),
            "retained_history_length": p["retained_length"],
            "valid_text_history_fraction": p["valid_text_fraction"],
            "valid_entity_history_fraction": p["valid_entity_fraction"],
            "no_supplied_history": p["no_supplied_history"],
            "no_causal_history": total == 0,
            "no_text_profile": not p["has_text"],
            "no_entity_profile": not p["has_entity"],
            "no_recent_profile": not p["has_recent"],
            "category_affinity": category_counts / max(1, total),
            "subcategory_affinity": subcategory_counts / max(1, total),
            "seen_category": category_counts > 0,
            "text_cosine": bundle.text[article_rows] @ p["text"],
            "entity_cosine": bundle.entity[article_rows] @ p["entity"],
            "long_term_cosine": bundle.text[article_rows] @ p["long_text"],
            "recent_cosine": bundle.text[article_rows] @ p["recent_text"],
            "text_match_valid": p["has_text"] & text_valid[idx],
            "entity_match_valid": p["has_entity"] & entity_valid[idx],
            "long_term_match_valid": bool(np.linalg.norm(p["long_text"]) > 0)
            & text_valid[idx],
            "recent_match_valid": p["has_recent"] & text_valid[idx],
            "previously_clicked": (pair_c[idx] > 0)
            | np.isin(article_rows, user.snapshot),
            "multi_interest_match_score": np.max(
                bundle.title[article_rows] @ p["interests"].T, axis=1
            )
            if p["group_count"]
            else np.zeros(len(idx)),
            "interest_group_count": p["group_count"],
            "selected_interest_group_count": p["selected_group_count"],
            "multi_interest_valid": bool(p["group_count"]) & title_valid[idx],
            "hours_since_previous_impression": (
                t - user.last_impression
            ).total_seconds()
            / 3600
            if user.last_impression is not None
            else 0,
            "first_visit": user.last_impression is None,
        }
        for name, value in dynamic.items():
            values[name][idx] = value
    return values

### 2.6 Assemble features and sample training pairs

`build_features` validates keys/timestamp/eligibility, combines article and user calculations, evaluates your registered custom features and casts the selected model columns. Optional slate context subtracts the actual candidate-set median interest score.

`sample_training_groups` makes a stable seeded draw of exposed non-clicks for each positive. A negative can occur in several positive groups; pointwise output keeps the union of unique pairs. Historical state still consumes the complete allowed earlier stream. Validation/test slates remain complete.

In [12]:
def build_features(
    requests,
    candidate_pairs,
    state_before_t,
    representation_bundle,
    feature_settings=DEFAULT_SETTINGS,
    candidate_source="retrieved",
):
    """Same input contract for logged or unlabelled retrieved candidates.

    Caller begins one complete timestamp, builds all feature requests, then calls
    finish with logged feedback. This function neither reads labels nor mutates state.
    """
    state = state_before_t
    bundle = representation_bundle
    cfg = feature_settings
    pairs, rows, t = prepare_feature_pairs(
        requests, candidate_pairs, state, bundle, cfg, candidate_source
    )
    values, validity = article_features(
        pairs, rows, state, bundle, cfg, t, candidate_source
    )
    values = add_user_features(values, pairs, rows, state, bundle, cfg, t, validity)
    result = pd.DataFrame(values)
    result["state_version"] = result.state_version.astype(np.int64)
    result["history_length"] = result.history_length.astype(np.int64)
    if cfg.context_features:
        grouped = result.groupby("impression_uid")
        valid = grouped.multi_interest_valid.transform("all")
        result["slate_context_valid"] = valid
        result["personalized_relative_advantage"] = (
            result.multi_interest_match_score
            - grouped.multi_interest_match_score.transform("median")
        ).where(valid, 0)
    active_contract = feature_schema(cfg)
    columns = active_contract["model_columns"]
    for name, specification in EXTRA_FEATURES.items():
        if name in columns:
            result[name] = specification["compute"](result, cfg)
    result[columns] = result[columns].astype(np.float32)
    if not np.isfinite(result[columns].to_numpy()).all():
        raise ValueError("Nonfinite model features")
    return result

In [13]:
def sample_training_groups(candidates, cfg=DEFAULT_SETTINGS):
    """Reproducible per-positive exposed negatives, independent of input row order."""
    groups = []
    selected = set()
    for uid, slate in candidates.groupby("impression_uid", sort=True):
        if slate.news_code.duplicated().any() or not slate.clicked.isin([0, 1]).all():
            raise ValueError("Sampling requires unique binary-labelled logged pairs")
        positives = sorted(map(int, slate.loc[slate.clicked.eq(1), "news_code"]))
        negatives = np.sort(
            slate.loc[slate.clicked.eq(0), "news_code"].to_numpy(np.int64)
        )
        if not positives:
            raise ValueError("Logged training impression has no positives")
        for positive in positives:
            seed = int.from_bytes(
                hashlib.sha256(f"{cfg.seed}:{int(uid)}:{positive}".encode()).digest()[
                    :8
                ],
                "little",
            )
            rng = np.random.default_rng(seed)
            sampled = sorted(
                map(
                    int,
                    rng.choice(
                        negatives,
                        min(cfg.negative_count, len(negatives)),
                        replace=False,
                    ),
                )
            )
            groups.append(
                {
                    "impression_uid": int(uid),
                    "positive_news_code": positive,
                    "negative_news_codes": sampled,
                }
            )
            selected.add((int(uid), positive))
            selected.update((int(uid), n) for n in sampled)
    return (
        pd.DataFrame(
            groups,
            columns=["impression_uid", "positive_news_code", "negative_news_codes"],
        ),
        selected,
    )

### 2.7 One shared worked example

This miniature stream uses hand-made teaching vectors. A user starts with sports article `N101` and science article `N102` in supplied history. Their current slate includes article 103, which they click; article 103 deliberately has no entity vector.

Read the first table as **features before feedback**, with the observed label shown beside them only for explanation. Then read the second table: the current click enters history **at the next timestamp**, giving history length **2 → 3** and prior clicks on 103 **0 → 1**. The label column is never passed to the feature function.

The input cell only creates this small in-memory fixture. It does not modify your selections, real representations or saved datasets.

In [14]:
DEMO_SETTINGS = replace(
    SETTINGS, text_dim=4, entity_dim=3, history_cap=3, half_life_min_exposures=2
)
demo_start = pd.Timestamp("2019-11-09 00:00:00")
demo_end = demo_start + pd.Timedelta(hours=24)
demo_index = pd.DataFrame(
    {
        "news_code": [101, 102, 103],
        "article_row": [1, 2, 3],
        "category": ["sports", "science", "sports"],
        "subcategory": ["football", "space", "football"],
        "category_code": [1, 2, 1],
        "subcategory_code": [1, 2, 1],
        "title_word_count": [5, 4, 6],
        "abstract_word_count": [12, 10, 0],
        "has_abstract": [True, True, False],
    }
)
demo_lookup = dict(zip(demo_index.news_code, demo_index.article_row))
demo_text = np.zeros((4, DEMO_SETTINGS.text_dim), dtype=np.float32)
demo_text[1, 0] = demo_text[2, 1] = 1
demo_text[3, :2] = 1 / np.sqrt(2)
demo_entity = np.zeros((4, DEMO_SETTINGS.entity_dim), dtype=np.float32)
demo_entity[1, 0] = demo_entity[2, 1] = 1
demo_bundle = SimpleNamespace(
    n=4,
    index=demo_index,
    lookup=demo_lookup,
    text=demo_text,
    title=demo_text.copy(),
    entity=demo_entity,
    meta=demo_index.set_index("article_row"),
    quality=pd.DataFrame(
        {
            "news_code": [101, 102, 103],
            "n_annotated_entities": [1, 1, 0],
            "entity_coverage": [1.0, 1.0, 0.0],
        }
    ).set_index("news_code"),
    rows=lambda codes: np.asarray([demo_lookup[int(c)] for c in codes], dtype=np.int32),
    eligible=lambda codes, t: np.full(
        len(codes), pd.Timestamp(t) >= demo_start, dtype=bool
    ),
)
demo_requests = pd.DataFrame(
    {
        "impression_uid": [500],
        "user_code": [7],
        "timestamp": [demo_start],
        "history": ["N101 N102"],
    }
)
demo_events = pd.DataFrame(
    {
        "impression_uid": [500] * 3,
        "user_code": [7] * 3,
        "timestamp": [demo_start] * 3,
        "news_code": [101, 102, 103],
        "clicked": [0, 0, 1],
        "split": ["train"] * 3,
    }
)

In [15]:
walkthrough_state = TemporalState(demo_bundle, demo_start, demo_end, DEMO_SETTINGS)
try:
    walkthrough_state.begin(demo_requests, demo_start)
    walkthrough_before = walkthrough_state.profile(7, demo_start)
    walkthrough_before_clicks = int(walkthrough_state.clicks[3])
    walkthrough_features = build_features(
        demo_requests,
        demo_events[["impression_uid", "news_code"]],
        walkthrough_state,
        demo_bundle,
        DEMO_SETTINGS,
        "retrieved",
    )
    walkthrough_display = walkthrough_features[
        [
            "news_code",
            "history_length",
            "text_cosine",
            "multi_interest_match_score",
            "prior_article_clicks",
            "entity_match_valid",
        ]
    ].merge(
        demo_events[["news_code", "clicked"]].rename(
            columns={"clicked": "observed_label_not_model_input"}
        ),
        on="news_code",
        validate="one_to_one",
    )
    display(walkthrough_display)
    walkthrough_state.finish(demo_requests, demo_events, demo_start, {"train"})
    next_time = demo_start + pd.Timedelta(hours=1)
    next_request = demo_requests.assign(impression_uid=501, timestamp=next_time)
    walkthrough_state.begin(next_request, next_time)
    walkthrough_after = walkthrough_state.profile(7, next_time)
    walkthrough_after_clicks = int(walkthrough_state.clicks[3])
    display(
        pd.DataFrame(
            [
                {
                    "phase": "current request, before feedback",
                    "history_length": walkthrough_before["history_length"],
                    "recent_rows": walkthrough_before["recent_rows"],
                    "prior_clicks_on_103": walkthrough_before_clicks,
                },
                {
                    "phase": "next request, after earlier feedback",
                    "history_length": walkthrough_after["history_length"],
                    "recent_rows": walkthrough_after["recent_rows"],
                    "prior_clicks_on_103": walkthrough_after_clicks,
                },
            ]
        )
    )
finally:
    walkthrough_state.close()

,news_code,history_length,text_cosine,multi_interest_match_score,prior_article_clicks,entity_match_valid,observed_label_not_model_input
0,101,2,0.707107,1.000000,0,1.0,0
1,102,2,0.707107,1.000000,0,1.0,0
2,103,2,1.000000,0.707107,0,0.0,1


,phase,history_length,recent_rows,prior_clicks_on_103
0,"current request, before feedback",2,[],0
1,"next request, after earlier feedback",3,[3],1


### 2.8 Compact verification

Consolidate the key guarantees here: the worked example excludes current feedback, sampled negatives are reproducible and belong to the logged slate, and the same **first real timestamp** produces identical model values with or without labels. Generic input/cache checks remain in the shared helpers.

The real-data parity check uses the same candidate set; optional slate-context values may differ when retrieval changes the set. These checks do not establish end-to-end recommendation quality.

In [16]:
assert walkthrough_before["history_length"] == 2 and walkthrough_before_clicks == 0
assert walkthrough_after["history_length"] == 3 and walkthrough_after_clicks == 1
example_groups, example_pairs = sample_training_groups(demo_events, DEMO_SETTINGS)
shuffled_groups, shuffled_pairs = sample_training_groups(
    demo_events.sample(frac=1, random_state=1), DEMO_SETTINGS
)
pd.testing.assert_frame_equal(example_groups, shuffled_groups)
assert example_pairs == shuffled_pairs
negative_ids = set(demo_events.loc[demo_events.clicked.eq(0), "news_code"])
assert all(set(ids) <= negative_ids for ids in example_groups.negative_news_codes)

check_bundle = Bundle(ROOT, SETTINGS)
check_state = TemporalState(check_bundle, *time_horizon(ROOT), SETTINGS)
try:
    check_time, check_requests, check_events = next(replay_batches(ROOT, SETTINGS))
    check_state.begin(check_requests, check_time)
    with_labels = build_features(
        check_requests, check_events, check_state, check_bundle, SETTINGS, "logged"
    )
    without_labels = build_features(
        check_requests,
        check_events[["impression_uid", "news_code"]],
        check_state,
        check_bundle,
        SETTINGS,
        "retrieved",
    )
    pd.testing.assert_frame_equal(
        with_labels[schema["model_columns"]], without_labels[schema["model_columns"]]
    )
    # Retrieve eligible history-only articles, not just stripped logged pairs.
    eligible_codes = check_bundle.availability.index[
        check_bundle.availability.available_from.le(check_time)
    ]
    novel_codes = [int(code) for code in eligible_codes if code not in set(check_events.news_code)][:3]
    if novel_codes:
        retrieved_pairs = pd.DataFrame({
            "impression_uid": [int(check_requests.impression_uid.iloc[0])] * len(novel_codes),
            "news_code": novel_codes,
        })
        retrieved_preview = build_features(check_requests, retrieved_pairs, check_state, check_bundle, SETTINGS, "retrieved")
        assert retrieved_preview.prior_article_exposures.eq(0).all()
        assert retrieved_preview.eligible.all()
        assert np.isfinite(retrieved_preview[schema["model_columns"]].to_numpy()).all()
        print("Retrieved history-only examples (no invented labels):")
        display(retrieved_preview[["news_code", "prior_article_exposures", "smoothed_prior_article_ctr", "text_cosine", "no_text_profile"]])
    else:
        print("No additional eligible history-only item at the first timestamp; synthetic regression tests cover this case.")
finally:
    check_state.close()
display(
    pd.DataFrame(
        {
            "check": [
                "Current feedback excluded",
                "Earlier feedback available next time",
                "Sampling reproducible after shuffle",
                "Sampled negatives were exposed",
                "Label-free parity on real inputs",
            ],
            "result": ["passed"] * 5,
        }
    )
)

Retrieved history-only examples (no invented labels):


,news_code,prior_article_exposures,smoothed_prior_article_ctr,text_cosine,no_text_profile
0,454,0,0.5,0.554446,0.0
1,9524,0,0.5,0.248535,0.0
2,13865,0,0.5,0.332105,0.0


,check,result
0,Current feedback excluded,passed
1,Earlier feedback available next time,passed
2,Sampling reproducible after shuffle,passed
3,Sampled negatives were exposed,passed
4,Label-free parity on real inputs,passed


## 3. Build or reopen the actual MIND feature dataset

The worked example above was synthetic and in memory; the parity check only read a small real timestamp. This cell uses your prepared inputs, real representation bundle and the **same functions/classes defined above**.

- The terminal runner defaults to a full build. For interactive execution, set `FULL_RUN` in the first code cell: `False` selects a 1,000-timestamp smoke check and `True` selects full replay. Terminal full/smoke flags take priority.
- Prequential mode updates state only after each timestamp. For frozen evaluation choose a target split and `Settings(update_mode="frozen")`; earlier splits still supply history.
- `REBUILD=False` loads a complete, verified compatible run. `REBUILD=True` replaces the selected run directory.

The build uses bounded Parquet batches and disk-backed pair history. A smoke result is not a full evaluation; it can legitimately have no validation/test partitions. The row counts printed below are written feature rows **after training sampling**.

**Contract v2:** full causal pooling and categorical metadata change the feature contract. Rebuild old v1 artifacts with `python run_all.py --from-stage 3 --to-stage 4 --force` before model handoff. Full-history row references add storage and memory; inspect the new cost report after rebuilding. Previous full-run results are not a validation of this revised contract.

In [17]:
# Terminal flags take priority; otherwise use FULL_RUN from the first code cell.
_runner_full_run = os.environ.get("NEWSREC_FULL_FEATURES")
use_full_run = FULL_RUN if _runner_full_run is None else _runner_full_run == "1"
MAX_TIMESTAMPS = (
    None if use_full_run else int(os.environ.get("NEWSREC_SMOKE_TIMESTAMPS", "1000"))
)
print(
    "Execution mode:",
    "full dataset" if use_full_run else f"smoke ({MAX_TIMESTAMPS:,} timestamps)",
)
TARGET_SPLIT = None  # Set validation/test with Settings(update_mode='frozen').
OVERWRITE = REBUILD or os.environ.get("NEWSREC_OVERWRITE_FEATURES") == "1"
run_dir = feature_run(
    ROOT,
    SETTINGS,
    target_split=TARGET_SPLIT,
    max_timestamps=MAX_TIMESTAMPS,
    rebuild=OVERWRITE,
    state_factory=TemporalState,
    feature_builder=build_features,
    negative_sampler=sample_training_groups,
    schema_builder=feature_schema,
)
print(run_dir)

run_manifest = json.loads((run_dir / "run_manifest.json").read_text())
display(
    pd.DataFrame(
        {
            "split": list(run_manifest["rows_written"]),
            "written_feature_rows": list(run_manifest["rows_written"].values()),
        }
    )
)

Execution mode: full dataset


Replaying 520,464 timestamp batches. ETA adapts to observed speed and candidate volume; final checks take additional time.


2,313/520,464 timestamp batches (0.4%); elapsed 31s; replay ETA ~2h 56m; finish ~2026-10-20 14:55 +08 (approx; final checks follow); cutoff 2019-11-09 01:19:17; emitted 20,981 pairs


4,560/520,464 timestamp batches (0.9%); elapsed 2m; replay ETA ~2h 57m; finish ~2026-10-20 14:57 +08 (approx; final checks follow); cutoff 2019-11-09 02:29:08; emitted 42,756 pairs


6,578/520,464 timestamp batches (1.3%); elapsed 2m; replay ETA ~3h 01m; finish ~2026-10-20 15:02 +08 (approx; final checks follow); cutoff 2019-11-09 03:17:10; emitted 66,579 pairs


8,776/520,464 timestamp batches (1.7%); elapsed 3m; replay ETA ~2h 57m; finish ~2026-10-20 14:57 +08 (approx; final checks follow); cutoff 2019-11-09 04:01:26; emitted 99,348 pairs


... (352 outputs left out)

520,000/520,464 timestamp batches (99.9%); elapsed 2h 49m; replay ETA ~6s; finish ~2026-10-20 15:55 +08 (approx; final checks follow); cutoff 2019-11-15 23:22:56; emitted 40,341,535 pairs


520,464/520,464 timestamp batches (100.0%); elapsed 2h 49m; replay complete; finalizing files and checks


/Users/bryan03/Desktop/BT4222-Group-6-/4_Feature_Engineering/artifacts/prequential-all


,split,written_feature_rows
0,train,13084988
1,validation,13192987
2,test,14085557


## 4. Audit the real saved outputs

The remaining tables come from `run_dir`, not the toy examples. Coverage describes **all logged exposures**, including training negatives omitted from feature calculation. Feature-quality reports describe **written rows**: sampled training pairs and complete validation/test slates. On the optimisation branch, outputs use a separate `-optimized` directory and phase timings are saved in `replay_timing.csv`; the audited v2 artifacts are preserved.

### 4.1 Representation and profile coverage

Start with all shown exposures. A low valid fraction indicates how often the ranker will rely on a missing-vector fallback. Article-vector coverage and user-profile coverage are different quantities. The complete CSV also contains clicked/new-user/new-article cohorts for later cold-start analysis.

In [18]:
coverage = pd.read_csv(run_dir / "coverage.csv")
display(
    coverage.loc[
        coverage.cohort.eq("shown_exposures"),
        [
            "split",
            "representation",
            "denominator",
            "valid",
            "fallback",
            "valid_fraction",
        ],
    ].reset_index(drop=True)
)
print("All saved coverage cohorts:", ", ".join(sorted(coverage.cohort.unique())))

,split,representation,denominator,valid,fallback,valid_fraction
0,test,entity,14085557,11115099,2970458,0.789113
1,test,entity_profile,14085557,14018207,67350,0.995219
2,test,text,14085557,14085557,0,1.000000
3,test,text_profile,14085557,14038172,47385,0.996636
4,train,entity,70314387,54801466,15512921,0.779378
5,train,entity_profile,70314387,69702971,611416,0.991305
6,train,text,70314387,70314387,0,1.000000
7,train,text_profile,70314387,69874693,439694,0.993747
8,validation,entity,13192987,10496534,2696453,0.795615
9,validation,entity_profile,13192987,13131850,61137,0.995366


All saved coverage cohorts: articles_unexposed_in_train, clicked_exposures, new_users_vs_train, no_causal_history, returning_users_vs_train, shown_exposures


### 4.2 Value ranges, missingness and numerical validity

The summary below helps identify broken values without dumping the full feature table. A high zero fraction can be expected for sparse behaviour or missing profiles; read it together with validity flags. Finite/range checks do not prove ranking quality or remove the need for downstream ablations.

The first table averages zero fractions within each feature group; it is a numerical summary, not a representation fallback rate.

In [19]:
quality = pd.read_csv(run_dir / "feature_quality.csv")
quality_by_group = quality.merge(
    contract[["name", "group"]],
    left_on="feature",
    right_on="name",
    how="left",
    validate="many_to_one",
)
zero_summary = (
    quality_by_group.groupby(["group", "split"])["zero_fraction"]
    .mean()
    .unstack("split")
)
print("Mean zero fraction by group and split:")
display(zero_summary.round(3))
print("Total nonfinite values:", int(quality.nonfinite_count.sum()))
display(
    quality.sort_values("zero_fraction", ascending=False)[
        ["split", "feature", "min", "max", "zero_fraction"]
    ].head(8)
)
assert quality.nonfinite_count.sum() == 0

Mean zero fraction by group and split:


split,test,train,validation
group,,,
content,0.075,0.082,0.073
freshness,0.500,0.500,0.500
half_life,0.899,0.876,0.917
history,0.503,0.497,0.503
match,0.234,0.244,0.230
multi_interest,0.003,0.006,0.003
popularity,0.249,0.248,0.248
repeat,0.787,0.784,0.776
request_context,0.500,0.500,0.500


Total nonfinite values: 0


,split,feature,min,max,zero_fraction
140,test,no_prior_article_exposure,0.0,1.0,0.999839
90,validation,no_prior_article_exposure,0.0,1.0,0.999818
40,train,no_prior_article_exposure,0.0,1.0,0.999649
64,validation,no_causal_history,0.0,1.0,0.996943
65,validation,no_text_profile,0.0,1.0,0.996943
115,test,no_text_profile,0.0,1.0,0.996636
114,test,no_causal_history,0.0,1.0,0.996636
66,validation,no_entity_profile,0.0,1.0,0.995366


### 4.3 Check what training sampling retained

Positive-group memberships and unique written negative pairs are different counts. A negative reused across two groups contributes two memberships but only one pointwise row. The discarded-negative fraction describes the reduction in written training negatives, not feedback discarded from history.

In [20]:
sampling_summary = json.loads((run_dir / "sampling_summary.json").read_text())
display(pd.DataFrame(sampling_summary.items(), columns=["sampling_measure", "value"]))

,sampling_measure,value
0,all_negatives,6.744539e+07
1,kept_unique_negatives,1.021599e+07
2,positive_groups,2.868998e+06
3,negative_group_memberships,1.088874e+07
4,discarded_unique_negative_fraction,8.485295e-01


### 4.4 Inspect one real request–article row

Read a small Arrow batch instead of loading the full feature dataset into pandas. The first table includes join keys and selected signals. The second annotates the first candidate’s model values using the same feature groups from section 1. Which missingness flags explain a zero value?

In [21]:
real_sample = feature_sample(run_dir, split="train", rows=3)
if real_sample.empty:
    print("This run contains no written training partition.")
else:
    display(
        real_sample[
            [
                "impression_uid",
                "news_code",
                "history_length",
                "text_cosine",
                "multi_interest_match_score",
                "article_age_hours",
                "no_text_profile",
                "has_half_life",
            ]
        ]
    )
    first_candidate = contract[["group", "name"]].copy()
    first_candidate["value"] = [
        real_sample.iloc[0][name] for name in first_candidate.name
    ]
    display(
        first_candidate.loc[
            (
                first_candidate.group.isin(["history", "match", "freshness"])
                | first_candidate.name.isin(EXTRA_FEATURES)
            ),
            :,
        ].reset_index(drop=True)
    )

,impression_uid,news_code,history_length,text_cosine,multi_interest_match_score,article_age_hours,no_text_profile,has_half_life
0,309963,117242,18,0.417849,0.402017,17.276667,0.0,0.0
1,309963,127836,18,0.259100,0.165245,16.636667,0.0,0.0
2,309963,43752,18,0.287412,0.121406,17.276667,0.0,0.0


,group,name,value
0,history,log_history_length,2.944439
1,history,retained_history_length,18.000000
2,history,valid_text_history_fraction,1.000000
3,history,valid_entity_history_fraction,0.888889
4,history,no_supplied_history,0.000000
5,history,no_causal_history,0.000000
6,history,no_text_profile,0.000000
7,history,no_entity_profile,0.000000
8,history,no_recent_profile,1.000000
9,match,category_affinity,0.111111


### 4.5 Runtime and resource cost

These are measurements recorded when this selected dataset was built. Loading cached outputs today does not rerun the historical build or replace its recorded runtime. Peak process RSS is sampled process memory, not total system memory; artifact size includes the saved feature-run files.

In [22]:
cost = pd.read_csv(run_dir / "feature_cost.csv")
display(
    cost[
        [
            "run",
            "wall_seconds",
            "sampled_peak_process_rss_mb",
            "artifact_bytes",
            "emitted_rows",
            "timestamp_batches",
        ]
    ]
)

,run,wall_seconds,sampled_peak_process_rss_mb,artifact_bytes,emitted_rows,timestamp_batches
0,prequential-all,10097.48278,4819.697664,2684540066,40363532,520464


## 5. Handoff to baselines, retrieval and re-ranking

| Saved output | How later work uses it |
|---|---|
| `feature_schema.json` | Select only `model_columns`; keep keys/diagnostics out of the model matrix. |
| `ranking_features_<split>/` | Read feature partitions by split and selected columns. |
| `labels_<split>/` | Join `clicked` by `(impression_uid, news_code)` for supervised training/evaluation. |
| `request_profiles_<split>/` | Reconstruct the exact causal history, masks and interest references for a request. |
| `training_negative_groups/` | Preserve positive-to-negative memberships for grouped losses. |
| Coverage, quality, sampling and cost reports | Report availability, numerical validity, sampling choices and computational cost. |
| `run_manifest.json`, `run_status.json` | Confirm completion, settings/dependencies, smoke/full scope and evaluation mode. |

For unlabelled retrieved candidates, reuse `build_features` with the same pre-feedback request state and representation settings. The function does not invent labels for unshown articles. Fit any downstream scaler, clusters or ranker using training data only.

**Limits to state in the report:** observed age is not publication age; unsupported decay is masked; the offline representation space is fitted on the training catalogue; evaluation must specify prequential versus frozen feedback. Runtime/coverage checks alone do not establish a ranking improvement or CTR uplift. Graph embeddings, interest clustering, ANN retrieval and learned rankers belong to later stages.

If you change the causal profile calculation, keep its repeated methods consistent with `3_03_user_profiles.ipynb`, then rebuild affected outputs. No rebuild is needed merely for editing explanations or displaying additional examples.

Teammates can import the same visible definitions with `from newsrec.ranking import TemporalState, build_features, feature_schema` (launch Python with `PYTHONPATH=src`). This adapter loads only trusted tagged definitions/selections from this notebook; it does not run cells, download data or rebuild artifacts. Restart Python after notebook changes.

These components support a retrieved-candidate smoke path; they do not constitute a trained, evaluated end-to-end recommender. Retrieval recall, complete-slate model evaluation, and feature-group ablations remain downstream work. Slate-relative context remains opt-in and must use the actual retrieved candidate set.

After a full build, run `PYTHONPATH=src python -m newsrec.sanity` for a bounded training logistic smoke check against popularity on complete validation slates. Its preprocessing is fitted only on the training sample; it never evaluates test labels. This diagnostic does not prove absence of leakage or calibrate sampled training probabilities.

## Optional appendix: additional demonstrations

The main walkthrough is complete. These independent examples provide more detail on vector pooling, half-life windows, optional slate context and custom-feature syntax. Skip them when you only need to select features and build the dataset. They do not modify production selections or saved artifacts.

### A. Valid-vector pooling

Two perpendicular valid vectors pool to a normalised direction equally close to both. The zero/padding row contributes nothing.

In [23]:
pooling_input = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], dtype=np.float32)
pooled, valid_count = normalised_mean(pooling_input, [1, 2, 0])
display(
    pd.DataFrame(pooling_input, columns=["coordinate_1", "coordinate_2"]).rename_axis(
        "row"
    )
)
display(
    pd.DataFrame(
        [
            {
                "pooled_vector": np.round(pooled, 4).tolist(),
                "valid_history_entries": valid_count,
                "L2_norm": round(float(np.linalg.norm(pooled)), 4),
            }
        ]
    )
)
assert valid_count == 2
np.testing.assert_allclose(pooled, [1 / np.sqrt(2), 1 / np.sqrt(2)], atol=1e-6)

,coordinate_1,coordinate_2
row,,
0,0.0,0.0
1,1.0,0.0
2,0.0,1.0


,pooled_vector,valid_history_entries,L2_norm
0,"[0.707099974155426, 0.707099974155426]",2,1.0


### B. Half-life windows and unsupported estimates

These are deliberately synthetic earlier hourly counts: 24/120 clicks/exposures in the early window versus 12/120 in the recent window. Another article has too little support, and another has no prior exposure. Altering the current unfinished hour must not change the estimate.

In [24]:
half_demo = TemporalState(demo_bundle, demo_start, demo_end, DEMO_SETTINGS)
try:
    half_demo.first[[1, 2]] = demo_start.value
    half_demo.hour_exposures[1, :6] = 20
    half_demo.hour_clicks[1, :6] = 4
    half_demo.hour_exposures[1, 12:18] = 20
    half_demo.hour_clicks[1, 12:18] = 2
    cutoff = demo_start + pd.Timedelta(hours=18)
    estimates = half_demo.half_life([1, 2, 3], cutoff)
    display(
        pd.DataFrame(
            estimates,
            columns=[
                "half_life_hours",
                "valid",
                "reason",
                "early_exposures",
                "recent_exposures",
            ],
        ).assign(news_code=[101, 102, 103])
    )
    half_demo.hour_exposures[1, 18] = 10000
    half_demo.hour_clicks[1, 18] = 9000
    assert half_demo.half_life([1], cutoff)[0] == estimates[0]
    print("Changing the unfinished hour leaves the estimate unchanged.")
finally:
    half_demo.close()

,half_life_hours,valid,reason,early_exposures,recent_exposures,news_code
0,12.719727,True,available,120,120,101
1,0.000000,False,low_support,0,0,102
2,0.000000,False,no_prior_exposure,0,0,103


Changing the unfinished hour leaves the estimate unchanged.


### C. Optional slate-relative matching

Temporarily enable context for the toy slate only. Relative advantage is the interest score minus this candidate set’s median; validity requires usable multi-interest matches for all candidates. The real-build settings remain unchanged.

In [25]:
context_settings = replace(DEMO_SETTINGS, context_features=True)
context_demo = TemporalState(demo_bundle, demo_start, demo_end, context_settings)
try:
    context_demo.begin(demo_requests, demo_start)
    context_features = build_features(
        demo_requests,
        demo_events[["impression_uid", "news_code"]],
        context_demo,
        demo_bundle,
        context_settings,
    )
    display(
        context_features[
            [
                "news_code",
                "multi_interest_match_score",
                "personalized_relative_advantage",
                "slate_context_valid",
            ]
        ]
    )
    np.testing.assert_allclose(
        context_features.personalized_relative_advantage,
        context_features.multi_interest_match_score
        - context_features.multi_interest_match_score.median(),
        atol=1e-6,
    )
finally:
    context_demo.close()

,news_code,multi_interest_match_score,personalized_relative_advantage,slate_context_valid
0,101,1.000000,0.000000,1.0
1,102,1.000000,0.000000,1.0
2,103,0.707107,-0.292893,1.0


### D. Custom-feature registration

This isolated example uses the same `group`, `definition` and `compute(frame, cfg)` format as `EXTRA_FEATURES`. The counts are synthetic and strictly earlier events. Copy/uncomment the corresponding entry in Part 1.0 and rebuild to include it in the real schema; this preview does not register it automatically.

In [26]:
custom_example = {
    "group": "custom",
    "definition": "past article clicks / (past article exposures + 1)",
    "compute": lambda frame, cfg: (
        frame["prior_article_clicks"] / (frame["prior_article_exposures"] + 1)
    ),
}
custom_input = pd.DataFrame(
    {
        "news_code": [101, 102, 103],
        "prior_article_clicks": [2, 0, 1],
        "prior_article_exposures": [10, 0, 3],
    }
)
custom_preview = custom_input.copy()
custom_preview["prior_click_share"] = custom_example["compute"](
    custom_input, DEMO_SETTINGS
)
display(custom_preview)
print(
    "Copy the prior_click_share entry into EXTRA_FEATURES to add it to the saved schema."
)

,news_code,prior_article_clicks,prior_article_exposures,prior_click_share
0,101,2,10,0.181818
1,102,0,0,0.000000
2,103,1,3,0.250000


Copy the prior_click_share entry into EXTRA_FEATURES to add it to the saved schema.
